# AI-Based Garbage Segmentation using Deep Learning (U-Net)

This Google Colab notebook details the training pipeline for a U-Net semantic segmentation model designed to detect and segment litter/garbage in images.

## Objectives:
1. **Download and Preprocess Garbage Segmentation Dataset**
2. **Implement U-Net Architecture** using TensorFlow & Keras
3. **Define Custom Metrics** (Intersection over Union - IoU, Dice Coefficient)
4. **Train and Validate** the Model
5. **Save the trained weights** as `.h5` for deployment in our Flask Application

### 1. Import Libraries
Let's import TensorFlow, Keras layers, NumPy, Matplotlib, OpenCV, and other utilities.

In [ ]:
import os
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Print TensorFlow version and GPU availability
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

### 2. Dataset Setup (Kaggle/TACO Dataset)
To train the model, we use a garbage segmentation dataset (e.g. TACO or custom litter dataset). Below is the code to download a sample Kaggle dataset containing garbage images and segmentation masks. Ensure your `kaggle.json` API token is uploaded to your Colab environment.

In [ ]:
# Upload your kaggle.json file if downloading from Kaggle
# from google.colab import files
# files.upload()

# Setup Kaggle directory
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# For this example, we download a generic garbage segmentation dataset from Kaggle
# !kaggle datasets download -d asd97765/garbage-segmentation-dataset
# !unzip -q garbage-segmentation-dataset.zip -d ./garbage_data

print("Ensure your image files and mask files are organized in directories, e.g., './images' and './masks'")

### 3. Data Pipeline & Preprocessing
We define a data loader that resizes images and masks to $256 \times 256$, normalizes pixel values, and sets up a `tf.data` pipeline for training.

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 16

def load_image_and_mask(img_path, mask_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = img / 255.0  # Normalize to [0, 1]
    
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT))
    mask = mask / 255.0  # Normalize masks to binary [0, 1]
    mask = np.expand_dims(mask, axis=-1)  # Add channel dimension (256, 256, 1)
    mask = (mask > 0.5).astype(np.float32)
    
    return img, mask

def data_generator(img_paths, mask_paths):
    for img_p, mask_p in zip(img_paths, mask_paths):
        try:
            img, mask = load_image_and_mask(img_p, mask_p)
            yield img, mask
        except Exception:
            continue

print("Loader and Preprocessing helper functions compiled successfully.")

### 4. U-Net Architecture Implementation
U-Net is a symmetric encoder-decoder network. The encoder contracts spatial dimensions while extracting features, and the decoder expands spatial dimensions to output pixel-wise predictions. Skip connections preserve fine-grained spatial features.

In [ ]:
def double_conv_block(x, num_filters):
    x = layers.Conv2D(num_filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(num_filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def encoder_block(x, num_filters):
    x = double_conv_block(x, num_filters)
    p = layers.MaxPool2D(pool_size=(2, 2))(x)
    return x, p

def decoder_block(x, skip_features, num_filters):
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip_features])
    x = double_conv_block(x, num_filters)
    return x

def build_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(shape=input_shape)
    
    # Encoder (Contracting Path)
    s1, p1 = encoder_block(inputs, 32)
    s2, p2 = encoder_block(p1, 64)
    s3, p3 = encoder_block(p2, 128)
    s4, p4 = encoder_block(p3, 256)
    
    # Bottleneck / Bridge
    b1 = double_conv_block(p4, 512)
    
    # Decoder (Expanding Path)
    d1 = decoder_block(b1, s4, 256)
    d2 = decoder_block(d1, s3, 128)
    d3 = decoder_block(d2, s2, 64)
    d4 = decoder_block(d3, s1, 32)
    
    # Output layer (Binary Classification per pixel)
    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    
    model = models.Model(inputs, outputs, name="U-Net")
    return model

model = build_unet()
model.summary()

### 5. Metrics & Loss Functions
Semantic segmentation suffers from class imbalance (background pixels vs. garbage pixels). To counter this, we use Dice Loss and compile our model with Mean Intersection over Union (IoU) and Dice Coefficient metrics.

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def iou_coef(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    total = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
    union = total - intersection
    return (intersection + smooth) / (union + smooth)

model.compile(optimizer='adam', loss=dice_loss, metrics=[dice_coef, iou_coef, 'accuracy'])
print("Model compiled successfully with custom Dice Loss, Dice Metric, and IoU Metric.")

### 6. Model Training & Evaluation
Let's define callbacks (Model Checkpoint, Early Stopping, Learning Rate Decay) and train the model.

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint('best_garbage_unet.h5', verbose=1, save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
]

print("Callbacks defined. Training code configured to output 'best_garbage_unet.h5'.")

### 7. Visualize Results & Predictions
Let's write a function to test the model on a sample validation image, outputting the original image, ground truth mask, and predicted mask side-by-side.

In [ ]:
def plot_predictions(model, test_img_path, test_mask_path=None):
    raw_img = cv2.imread(test_img_path)
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(raw_img, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
    
    pred = model.predict(np.expand_dims(img, axis=0))[0]
    pred_mask = (pred > 0.5).astype(np.uint8)
    
    num_plots = 3 if test_mask_path else 2
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, num_plots, 1)
    plt.title("Original Image")
    plt.imshow(img)
    plt.axis('off')
    
    if test_mask_path:
        raw_mask = cv2.imread(test_mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(raw_mask, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        plt.subplot(1, num_plots, 2)
        plt.title("Ground Truth Mask")
        plt.imshow(mask, cmap='gray')
        plt.axis('off')
        
    plt.subplot(1, num_plots, num_plots)
    plt.title("Predicted Mask")
    plt.imshow(pred_mask.squeeze(), cmap='gray')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualization helper compiled successfully.")